©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、機械学習モデル（SVM）に対するハイパーパラメータ調整手法を実践します。scikit-learnのGridSearchCVとRandomizedSearchCVを用いて、最適なパラメータ組み合わせを探索し、それぞれの手法の特徴や結果の違いを比較・確認します。

# 【3-2】ハイパーパラメータの調整

ここでは、乳がんのデータセットを用いて、**ハイパーパラメータの調整**について理解します．


**【目標】<font color="red">ハイパーパラメータの調整手法を実装できる．</font>**

## ０．各ライブラリを読み込む

In [ ]:
# データ加工・処理・分析モジュール
import pandas as pd
import numpy as np

## １．データを読み込む

In [ ]:
# 乳がんのデータセットを読み込みます
from sklearn import datasets
data = datasets.load_breast_cancer()

データを pandas ライブラリの DataFrame形式に変換します．

In [ ]:
# データをDataFrame形式に変換します
data_df = pd.DataFrame(data.data, columns=data.feature_names)
data_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## ２．SVMに"グリッドサーチ"を用いて予測する

グリッドサーチを用いて、最適なハイパーパラメータを探索してみます．

学習に用いるデータと評価に用いるデータに８：２の割合で分割します．

In [ ]:
from sklearn.model_selection import train_test_split
# 学習データと評価データに分割します
X_train, X_test, y_train, y_test = train_test_split(data_df, data.target, test_size=0.2, random_state=42)

次にグリッドサーチの範囲を設定していきます．

このリスト内に記入した数値から最も精度が良くなる組み合わせを探してくます．

==== 参考 ====

C：サポートベクトルマシンの正則化係数．値が大きいほど厳密に分類する．

gamma：マージンの曲率．値が大きいほど複雑な境界を作成する．

In [ ]:
# ハイパーパラメータのリストを設定します
param_grid = {"C":[0.1,0.5,1],
              "gamma":[0.1,0.5,1]}

scikit-learn ライブラリの GridSearchCV メソッドを用いることでグリッドサーチを行うことができます．

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn import svm

# サポートベクトルマシンを定義します
model = svm.SVC(kernel='linear')

# ハイパーパラメータの探索を実行します
grid_search = GridSearchCV(model,param_grid,verbose=1,cv=2)

# グリッドサーチを行います
grid_search.fit(X_train,y_train)

Fitting 2 folds for each of 9 candidates, totalling 18 fits


GridSearchCV(cv=2, estimator=SVC(kernel='linear'),
             param_grid={'C': [0.1, 0.5, 1], 'gamma': [0.1, 0.5, 1]},
             verbose=1)

※ cv という値を設定することで、より細かいクロスバリデーションを設定できるが、どんどん処理時間が長くなる．

### 最適なパラメータと評価の確認

In [ ]:
# 最適なハイパーパラメータを表示します
grid_search.best_params_

{'C': 1, 'gamma': 0.1}

In [ ]:
# テストデータによる評価を行います
grid_search.score(X_test,y_test)

0.956140350877193

## ３．SVMに"ランダムサーチ"を用いて予測する

ランダムサーチを用いて、最適なハイパーパラメータを探索してみます．

次にランダムサーチの範囲を設定していきます．

ランダムサーチは確率分布をパラメータの範囲として設定することができます．

このリスト内で設定した範囲からランダムに組み合わせを作成して、探してくます．

In [ ]:
import scipy

# ハイパーパラメータのリストを設定します
param_random = {"C":scipy.stats.uniform(0.1, 1),
                "gamma":scipy.stats.uniform(0.1, 1)}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# サポートベクトルマシンを定義します
model = svm.SVC(kernel='linear')

# ハイパーパラメータの探索を実行します
random_search = RandomizedSearchCV(model,param_random,verbose=1,cv=2,n_iter=10)

# グリッドサーチを行います
random_search.fit(X_train,y_train)

Fitting 2 folds for each of 10 candidates, totalling 20 fits


RandomizedSearchCV(cv=2, estimator=SVC(kernel='linear'),
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7faf1493e4b0>,
                                        'gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7faf13e78da0>},
                   verbose=1)

### 最適なパラメータと評価の確認

In [ ]:
# 最適なハイパーパラメータを表示します
random_search.best_params_

{'C': np.float64(1.0054920837775414), 'gamma': np.float64(0.9035112438364499)}

In [ ]:
# テストデータによる評価を行います
random_search.score(X_test,y_test)

0.956140350877193

## 🔧 実践問題1：カーネルの種類も含めたグリッドサーチを実装する

上のコードでは `kernel='linear'` を固定してC, gammaだけを探索しました。
しかし、SVMのカーネルの選択自体も精度に大きく影響します。

| カーネル | 特徴 | `gamma`の影響 |
|:---|:---|:---|
| `'linear'` | 線形分離。高次元データに強い | gammaは使われない |
| `'rbf'` | 非線形境界を学習できる。最も汎用的 | 大きいほど複雑な境界 |
| `'poly'` | 多項式カーネル。`degree`で次数を指定 | 影響あり |

`kernel` もグリッドサーチの探索対象に加えることで、最適なカーネルとパラメータの組み合わせを同時に探索できます。

---

**問題：** 以下のコードの `______` を埋めて、カーネルの種類（`'linear'`, `'rbf'`）も含めた探索を実行し、`cv_results_` から全探索結果を表示してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
<code>param_grid</code> の辞書にキー <code>'kernel'</code> を追加すれば、カーネルも探索対象になります。<code>cv_results_</code> にはすべての組み合わせのスコアが記録されています。
</blockquote>

</details>

<br/>


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn import svm
import pandas as pd

# カーネルの種類も探索対象に加える
param_grid2 = {
    '______': ['linear', '______'],  # カーネルの種類
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1]
}

# SVMを定義（kernelは固定しない）
model2 = svm.______()  # デフォルト設定のSVM

# グリッドサーチを実行（5分割交差検証）
grid_search2 = GridSearchCV(model2, param_grid2, cv=______, verbose=1)
grid_search2.fit(X_train, y_train)

# 最適パラメータとテスト精度を表示
print(f'最適パラメータ: {grid_search2.______}')
print(f'テスト精度: {grid_search2.score(X_test, y_test):.3f}')

# 全探索結果をDataFrameで表示（上位5件）
results = pd.DataFrame(grid_search2.cv_results_)
results = results.sort_values('rank_test_score')
print('\n--- 探索結果（上位5件） ---')
print(results[['params', 'mean_test_score', 'rank_test_score']].head())


<details><summary>解答例</summary>

```python
param_grid2 = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1]
}

model2 = svm.SVC()
grid_search2 = GridSearchCV(model2, param_grid2, cv=5, verbose=1)
grid_search2.fit(X_train, y_train)

print(f'最適パラメータ: {grid_search2.best_params_}')
print(f'テスト精度: {grid_search2.score(X_test, y_test):.3f}')
```

- `'kernel'` を `param_grid` に追加すると、カーネルの種類も組み合わせ探索の対象になります。2カーネル × 3C × 3gamma = **18通り**を探索します
- `svm.SVC()` をデフォルト設定で定義し、カーネルは `GridSearchCV` 側で切り替えます。上のコードのように `kernel='linear'` と固定するとグリッドサーチでカーネルを変えられません
- `cv=5` は5分割交差検証です。上のコードの `cv=2` より分割数が多い分、精度の見積もりが安定しますが計算時間は増えます
- `best_params_` で最適な組み合わせを、`cv_results_` で全組み合わせのスコアを確認できます。`'linear'` と `'rbf'` でどちらが良いか結果を見てみてください
- linearカーネルでは `gamma` は実際には使われません（無視されます）が、探索自体はエラーにならず実行されます
</details>


## 🔧 実践問題2：交差検証のスコアを手動で実装してグリッドサーチの仕組みを理解する

上のコードでは `GridSearchCV` が内部で自動的に交差検証を行っていました。
しかし、交差検証の仕組みを理解するためには、手動で実装してみることが重要です。

交差検証（K-Fold Cross Validation）の手順：
1. データをK個のグループ（fold）に分割する
2. K回の学習を行い、毎回1つのfoldをテスト用、残りを学習用にする
3. K回のスコアの**平均**を最終的なスコアとする

scikit-learnの `KFold` クラスを使うと、データのインデックスをK分割できます。

---

**問題：** 以下のコードの `______` を埋めて、3分割交差検証を手動で実装し、`GridSearchCV(cv=3)` の結果と一致することを確認してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
<code>KFold</code> の <code>split(X)</code> は各foldの <code>(train_index, test_index)</code> を順に返すイテレータです。
</blockquote>

</details>

<br/>


In [ ]:
from sklearn.model_selection import KFold
from sklearn import svm
import numpy as np

# X_train, y_trainをNumPy配列に変換（DataFrameのままだとインデックスでハマりやすい）
X_np = np.array(X_train)
y_np = np.array(y_train)

# 3分割交差検証を手動で実装
kf = KFold(n_splits=______, shuffle=False)

C_value = 1.0
gamma_value = 0.1
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.______(X_np)):
    # foldごとにデータを分割
    X_fold_train = X_np[train_idx]
    y_fold_train = y_np[train_idx]
    X_fold_val = X_np[______]
    y_fold_val = y_np[______]

    # SVMを学習
    clf = svm.SVC(C=C_value, gamma=gamma_value, kernel='rbf')
    clf.______(X_fold_train, y_fold_train)

    # バリデーションデータで精度を計算
    score = clf.______(X_fold_val, y_fold_val)
    scores.append(score)
    print(f'  Fold {fold+1}: accuracy = {score:.4f}')

print(f'\n手動CV平均スコア: {np.______(scores):.4f}')

# GridSearchCVの結果と比較
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(
    svm.SVC(C=C_value, gamma=gamma_value, kernel='rbf'),
    X_np, y_np, cv=3
)
print(f'cross_val_score: {cv_scores.mean():.4f}')
print(f'→ 一致していれば交差検証の仕組みを正しく理解できています')


<details><summary>解答例</summary>

```python
kf = KFold(n_splits=3, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np)):
    X_fold_val = X_np[val_idx]
    y_fold_val = y_np[val_idx]

    clf.fit(X_fold_train, y_fold_train)
    score = clf.score(X_fold_val, y_fold_val)

print(f'手動CV平均スコア: {np.mean(scores):.4f}')
```

- `KFold(n_splits=3)` はデータを3等分します。`shuffle=False` にすると順番通りに分割するので、結果が再現可能です
- `kf.split(X_np)` は各foldの `(train_index, val_index)` を返します。このインデックスで配列をスライスしてfoldごとのデータを取り出します
- `val_idx` を使ってバリデーションデータ `X_fold_val`, `y_fold_val` を取り出します。ここが**テストデータ（X_test）ではない**ことがポイントです。CVは学習データの中で分割します
- `clf.fit()` で学習、`clf.score()` で精度を算出し、3回分の平均 `np.mean(scores)` が交差検証スコアです
- `cross_val_score` の結果と一致すれば、`GridSearchCV` が内部でやっていることと同じ処理を手動で再現できたことになります
</details>
